In [ ]:
import os

# Path to the trace directory
trace_dir = "/trace/2026-10-08_14-10-18"

# Topics of interest - begin_topic can be a list to find the earliest occurrence
begin_topic = [
    "/drivers/ouster_lidar_fl/points/cloudini",
    "/drivers/ouster_lidar_fr/points/cloudini",
    "/drivers/ouster_lidar_rl/points/cloudini",
    "/drivers/ouster_lidar_rr/points/cloudini",
]
end_topic = [
    "/planning/trajectory_optimization/trajectory",
]

max_num_analyze = None  # Set to None to analyze all

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
ros_distro = os.environ.get("ROS_DISTRO")
sys.path.insert(0, f'/docker-ros/ws/install/tracetools_analysis/lib/python3.12/site-packages')

from tracetools_analysis.loading import load_file
from tracetools_analysis.processor.ros2 import Ros2Handler
from tracetools_analysis.utils.ros2 import Ros2DataModelUtil

In [ ]:
# Load and process trace data
events_list = load_file(str(trace_dir))
handler = Ros2Handler.process(events_list)
data_util = Ros2DataModelUtil(handler.data)

# Map all publisher and subscription handles to their topics
rcl_publisher_to_topic_map = {}
for rcl_publisher, row in handler.data.rcl_publishers.iterrows():
    rcl_publisher_to_topic_map[rcl_publisher] = row['topic_name']
rcl_subscription_to_topic_map = {}
for rcl_subscription, row in handler.data.rcl_subscriptions.iterrows():
    rcl_subscription_to_topic_map[rcl_subscription] = row['topic_name']

# Map all rcl publisher and subscription handles to their rmw handles
rcl_publisher_to_rmw_map = {}
for rcl_publisher, row in handler.data.rcl_publishers.iterrows():
    rcl_publisher_to_rmw_map[rcl_publisher] = row['rmw_handle']
rcl_subscription_to_rmw_map = {}
for rcl_subscription, row in handler.data.rcl_subscriptions.iterrows():
    rcl_subscription_to_rmw_map[rcl_subscription] = row['rmw_handle']

# Map all rmw publisher and subscription handles to their topics
rmw_publisher_to_topic_map = {}
for rcl_publisher, topic in rcl_publisher_to_topic_map.items():
    rmw_handle = rcl_publisher_to_rmw_map[rcl_publisher]
    rmw_publisher_to_topic_map[rmw_handle] = topic
rmw_subscription_to_topic_map = {}
for rcl_subscription, topic in rcl_subscription_to_topic_map.items():
    rmw_handle = rcl_subscription_to_rmw_map[rcl_subscription]
    rmw_subscription_to_topic_map[rmw_handle] = topic

# Map all rcl publishers to their subscriptions via annotated message links
rcl_periodic_publisher_to_subscriptions_map = {}
for _, row in handler.data.message_links_periodic_async.iterrows():
    for rcl_publisher in row['pubs']:
        if rcl_publisher in rcl_periodic_publisher_to_subscriptions_map:
            print(f"Warning: Overwriting more recent periodic message link for rcl_publisher: {rcl_publisher}")
        rcl_periodic_publisher_to_subscriptions_map[rcl_publisher] = set()
        for rcl_subscription in row['subs']:
            rcl_periodic_publisher_to_subscriptions_map[rcl_publisher].add(rcl_subscription)
rcl_partial_publisher_to_subscriptions_map = {}
for _, row in handler.data.message_links_partial_sync.iterrows():
    for rcl_publisher in row['pubs']:
        if rcl_publisher in rcl_partial_publisher_to_subscriptions_map:
            print(f"Warning: Overwriting more recent partial message link for rcl_publisher: {rcl_publisher}")
        rcl_partial_publisher_to_subscriptions_map[rcl_publisher] = set()
        for rcl_subscription in row['subs']:
            rcl_partial_publisher_to_subscriptions_map[rcl_publisher].add(rcl_subscription)

In [ ]:
def timestamp_to_readable(timestamp_ns):
    """Convert nanoseconds since epoch to HH:MM:SS.nnnnnnnnn format"""
    dt = pd.to_datetime(timestamp_ns, unit='ns', utc=True)
    # Get the time part and the nanosecond fraction
    time_str = dt.strftime('%H:%M:%S')
    nanoseconds = timestamp_ns % 1_000_000_000
    return f"{time_str}.{nanoseconds:09d}"

def get_dependencies_of_publisher(rcl_publish_instance):
    rcl_publisher = rcl_publish_instance.publisher_handle
    
    # Option 1: Message is published periodically depending on specified subscriptions (annotated with TRACETOOLS_TRACEPOINT() in source code)
    if rcl_publisher in rcl_periodic_publisher_to_subscriptions_map:
        subscriptions = rcl_periodic_publisher_to_subscriptions_map[rcl_publisher]
        print(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- periodic async <-- {[rcl_subscription_to_topic_map[sub] for sub in subscriptions]}")
    
    # Option 2: Message is published when all specified subscriptions have taken messages (annotated with TRACETOOLS_TRACEPOINT() in source code)
    elif rcl_publisher in rcl_partial_publisher_to_subscriptions_map:
        subscriptions = rcl_partial_publisher_to_subscriptions_map[rcl_publisher]
        print(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- partial sync <-- {[rcl_subscription_to_topic_map[sub] for sub in subscriptions]}")

    # Option 3: No annotated message links found for this publisher. Check if message is published during a subscription callback.    
    else:
        # check if published during a subscription callback
        rcl_callbacks_parallel = handler.data.callback_instances[
            (handler.data.callback_instances.timestamp.astype(np.int64) <= rcl_publish_instance.timestamp) &
            (handler.data.callback_instances.timestamp.astype(np.int64) + handler.data.callback_instances.duration.astype(np.int64) >= rcl_publish_instance.timestamp) &
            (handler.data.callback_instances.pid == rcl_publish_instance.pid) &
            (handler.data.callback_instances.tid == rcl_publish_instance.tid) &
            (handler.data.callback_instances.procname == rcl_publish_instance.procname)
        ]

        if rcl_callbacks_parallel.empty:
            print(f"    ❌ {rcl_publisher_to_topic_map[rcl_publisher]} <-- no dependencies found")
            subscriptions = []
        elif len(rcl_callbacks_parallel) > 1:
            raise ValueError(f"❌ Could not identify single callback for rcl_publisher 0x{rcl_publisher:x} at timestamp {rcl_publish_instance.timestamp}")
        else:
            rcl_callback_object = handler.data.callback_objects[handler.data.callback_objects.callback_object == rcl_callbacks_parallel.iloc[0].callback_object]
            rcl_sub_handles = handler.data.subscription_objects[handler.data.subscription_objects.index == rcl_callback_object.index[0]].subscription_handle
            subscriptions = None
            for rcl_sub_handle in rcl_sub_handles:
                if rcl_subscription_to_topic_map[rcl_sub_handle] != '/clock':
                    subscriptions = {rcl_sub_handle}
            if not subscriptions:
                raise ValueError(f"❌ Could not identify subscription for callback object {rcl_callback_object.iloc[0].callback_object}")
            print(f"    {rcl_publisher_to_topic_map[rcl_publisher]} <-- callback <-- {[rcl_subscription_to_topic_map[sub] for sub in subscriptions]}")
    
    return subscriptions

def get_latest_taken_message_for_subscription(rcl_subscription, timestamp):
    taken_messages = handler.data.rmw_take_instances[
        (handler.data.rmw_take_instances['subscription_handle'] == rcl_subscription_to_rmw_map[rcl_subscription]) &
        (handler.data.rmw_take_instances['timestamp'] <= timestamp)
    ]
    if taken_messages.empty:
        return None
    latest_taken_message = taken_messages.loc[taken_messages['timestamp'].idxmax()]
    return latest_taken_message

def get_published_for_taken_message(rmw_take_instance):
    taken_topic = rmw_subscription_to_topic_map[rmw_take_instance.subscription_handle]
    topic_publishers = [rmw_publisher for rmw_publisher, topic in rmw_publisher_to_topic_map.items() if topic == taken_topic]
    published_messages = handler.data.rmw_publish_instances[
        (handler.data.rmw_publish_instances['publisher_handle'].isin(topic_publishers)) &
        (handler.data.rmw_publish_instances['timestamp'] == rmw_take_instance.source_timestamp)
    ]
    if published_messages.empty:
        return None
    if len(published_messages) > 1:
        raise ValueError("❌ Could not identify single published for taken message")
    return published_messages.iloc[0]

def backpropagate_messages(rcl_publish_instance_end, begin_topics, time):
    if isinstance(begin_topics, str):
        begin_topics = [begin_topics]
    
    rcl_publish_topic = rcl_publisher_to_topic_map[rcl_publish_instance_end.publisher_handle]
    print(f"  ▶️ Analyzing publication on topic {rcl_publish_topic}")

    # collect processing durations between inputs and publication
    processing_durations_iteration[rcl_publish_topic] = {}

    # identify all subscriptions that are input to this publication
    rcl_subs = get_dependencies_of_publisher(rcl_publish_instance_end)
    
    for rcl_sub in rcl_subs:
        current_topic = rcl_subscription_to_topic_map[rcl_sub]
        print(f"     🔀 Following subscription on topic {current_topic}")
        
        rmw_take_instance = get_latest_taken_message_for_subscription(rcl_sub, rcl_publish_instance_end.timestamp)
        if rmw_take_instance is None:
            print(f"      ❌ rmw_take: No taken message found for subscribed topic {current_topic} at timestamp {timestamp_to_readable(rcl_publish_instance_end.timestamp)}")
            continue
        processing_durations_iteration[rcl_publish_topic][current_topic] = rcl_publish_instance_end.timestamp - rmw_take_instance.timestamp
        
        if handler.data.rmw_payload_instances.empty:
            print(f"      ❌ rmw_payload: No payload instance found for taken message of subscribed topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
            payload_size = None
        else:
            rmw_payload_instance = handler.data.rmw_payload_instances[
                (handler.data.rmw_payload_instances['subscription_handle'] == rmw_take_instance.subscription_handle) &
                (handler.data.rmw_payload_instances['message'] == rmw_take_instance.message)
            ]
            if rmw_payload_instance.empty:
                print(f"      ❌ rmw_payload: No payload instance found for taken message of subscribed topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
                payload_size = None
            elif len(rmw_payload_instance) > 1:
                print(f"      ❌ rmw_payload: Multiple payload instances found for taken message of subscribed topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
                payload_size = None
            else:
                payload_size = rmw_payload_instance.iloc[0]['payload_size']
        payload_sizes_iteration[current_topic] = payload_size

        print(f"      ✅ rmw_take: Message ({payload_size} bytes) taken at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
        
        if current_topic in begin_topics:
            print(f"    🏁 Reached begin topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
            # Update begin_time only if it's None or if we found an earlier time
            if time['begin_time'] is None or rmw_take_instance.timestamp < time['begin_time']:
                time['begin_time'] = rmw_take_instance.timestamp
                time['begin_topic_found'] = current_topic
            continue

        rmw_publish_instance = get_published_for_taken_message(rmw_take_instance)
        if rmw_publish_instance is None:
            print(f"      ❌ rmw_publish: No matching published message found for subscribed topic {current_topic} at timestamp {timestamp_to_readable(rmw_take_instance.timestamp)}")
            continue
        print(f"      ✅ rmw_publish: Message published at timestamp {timestamp_to_readable(rmw_publish_instance.timestamp)}")

        rcl_publish_instances = handler.data.rcl_publish_instances[
            (handler.data.rcl_publish_instances.message == rmw_publish_instance.message) &
            (handler.data.rcl_publish_instances.timestamp <= rmw_publish_instance.timestamp)
        ]
        rcl_publish_instance = rcl_publish_instances.loc[rcl_publish_instances.timestamp.idxmax()]
        if rcl_publish_instance.empty:
            raise ValueError(f"❌ rcl_publish: Could not identify single rcl_publish_instance for rmw_publish_instance 0x{rmw_publish_instance:x}")
        print(f"      ✅ rcl_publish: Message published at timestamp {timestamp_to_readable(rcl_publish_instance.timestamp)}")

        # collect transfer durations between published and taken messages
        transfer_durations_iteration[current_topic] = rmw_take_instance.timestamp - rcl_publish_instance.timestamp

        backpropagate_messages(rcl_publish_instance, begin_topics, time)


In [ ]:
critical_path_durations = {}
transfer_durations = {}
processing_durations = {}
payload_sizes = {}
incomplete_paths = []

# get all publishers of end_topic
rcl_publishers_end_topic = handler.data.rcl_publishers[handler.data.rcl_publishers['topic_name'].isin(end_topic)]
if len(rcl_publishers_end_topic) == 0:
    raise ValueError(f"No rcl_publishers found for '{end_topic}'")
elif len(rcl_publishers_end_topic) > 1:
    print(f"Warning: Multiple rcl_publishers found for '{end_topic}'. Will analyze all of them separately.")

for rcl_publisher, row in rcl_publishers_end_topic.iterrows():
    print(f"Analyzing critical path for end topic '{row.topic_name}' published by rcl_publisher '0x{rcl_publisher:x}'")

    # Find all published messages by this rcl_publisher
    rcl_publish_instances_end_topic = handler.data.rcl_publish_instances[handler.data.rcl_publish_instances['publisher_handle'] == rcl_publisher]
    print(f" Found {len(rcl_publish_instances_end_topic)} published messages for rcl_publisher '0x{rcl_publisher:x}'")
    num_analyzed = 0
    for pub_instance_idx, pub_instance_row in rcl_publish_instances_end_topic.iterrows():
        time = {'end_time': pub_instance_row.timestamp, 'begin_time': None, 'begin_topic_found': None}
        print(f"  {num_analyzed}: Analyzing rcl_publish_instance 0x{pub_instance_idx:x} at timestamp {timestamp_to_readable(time['end_time'])}")
        if num_analyzed == 61:
            pass
        transfer_durations_iteration = {}
        processing_durations_iteration = {}
        payload_sizes_iteration = {}
        rcl_taken_begin = backpropagate_messages(pub_instance_row, begin_topic, time)
        if time['begin_time'] is None:
            print(f"   Could not find begin time for end time {timestamp_to_readable(time['end_time'])}")
            incomplete_paths.append(time['end_time'])
            continue
        transfer_durations[time['end_time']] = transfer_durations_iteration
        processing_durations[time['end_time']] = processing_durations_iteration
        payload_sizes[time['end_time']] = payload_sizes_iteration
        duration = time['end_time'] - time['begin_time']
        print(f"  ✅ Critical path duration {duration/1e6} ms (from {time['begin_topic_found']})")
        critical_path_durations[time['end_time']] = duration
        num_analyzed += 1
        if max_num_analyze and num_analyzed >= max_num_analyze:
            print(f"Reached max_num_analyze limit of {max_num_analyze}. Stopping analysis for rcl_publisher '0x{rcl_publisher:x}'")
            break

## Compute Critical Path Durations

Convert the causal pairs (LiDAR callback → Ackermann publish) to duration measurements.

In [ ]:
if len(critical_path_durations) == 0:
    raise RuntimeError("No critical path flows found. The message links may not capture the causality.")

# Extract durations
timestamps_ns = np.fromiter(critical_path_durations.keys(), dtype=float)
durations_ms = np.array(list(critical_path_durations.values())) / 1e6

print(f"Could not follow {len(incomplete_paths)} incomplete paths: \n  - {', '.join([timestamp_to_readable(incomplete_path) for incomplete_path in incomplete_paths])}")
print(f"Computed {len(durations_ms)} critical-path durations")
print(f"Duration range: {durations_ms.min():.2f} ms to {durations_ms.max():.2f} ms")
print(f"Mean duration: {durations_ms.mean():.2f} ms")
print(f"Median duration: {np.median(durations_ms):.2f} ms")


## Analyze Message Flow

Plot histogram of durations and durations over end time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(durations_ms, bins=50, color="#4C72B0", edgecolor="white")
ax.set_title(f"Critical Path Duration \nFrom:\n{',\n'.join(begin_topic)}\nTo:\n{end_topic}")
ax.set_xlabel("Duration (ms)")
ax.set_ylabel("Count")
ax.grid(True, alpha=0.2)

summary = pd.Series(durations_ms).describe(percentiles=[0.5, 0.9, 0.99])
print(summary)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

# Convert timestamps from ms to datetime in Berlin timezone
timestamps_datetime = pd.to_datetime(timestamps_ns, unit='ns', utc=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timestamps_datetime, durations_ms, marker='o', markersize=3, linestyle='-', linewidth=0.5, color="#4C72B0")
ax.set_title("Critical Path Duration Over Time")
ax.set_xlabel("End Time")
ax.set_ylabel("Duration (ms)")
ax.grid(True, alpha=0.2)

# Format x-axis with time including milliseconds
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S.%f', tz='UTC'))
# Increase the number of ticks
ax.xaxis.set_major_locator(MaxNLocator(nbins=20))
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Analyze Transfer Durations per Topic and Processing Durations per Node

- **Message Trasfer:** Duration between a message being published by a node (`rcl_publish`) and the same message being taken by a subsequent node (`rmw_take`).
- **Node Processing:** Durations between each input message being taken by a node (`rmw_take`) and a message being published by the same node (`rcl_publish`) based on this input data.
- **Node Processing (shortest):** Duration between the _last required input message_ being taken by a node (`rmw_take`) and a message being published by the same node (`rcl_publish`) based on this input data.


In [ ]:
# Visualize transfer_durations and processing_durations
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 16))

# Plot 1: Transfer Durations Boxplot
# Structure: transfer_durations = {timestamp: {topic: duration_ns}}
# Aggregate durations per topic across all test runs
topic_durations_map = {}
for timestamp, topics_dict in transfer_durations.items():
    for topic, duration_ns in topics_dict.items():
        if topic not in topic_durations_map:
            topic_durations_map[topic] = []
        topic_durations_map[topic].append(duration_ns / 1e6)  # Convert to ms

# Sort topics by median duration
sorted_topics = sorted(topic_durations_map.items(), 
                       key=lambda x: np.median(x[1]), reverse=True)

# Limit to top topics to keep visualization readable
max_topics = 15
sorted_topics = sorted_topics[:max_topics]

topic_labels = [topic.split('/')[-1] if len(topic) > 40 else topic 
                for topic, _ in sorted_topics]
boxplot_data = [durations for _, durations in sorted_topics]

# Create boxplot
bp1 = ax1.boxplot(boxplot_data, vert=False, patch_artist=True,
                    showmeans=True, meanline=False,
                    medianprops=dict(color='red', linewidth=2),
                    meanprops=dict(marker='D', markerfacecolor='blue', markersize=6),
                    boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                    whiskerprops=dict(color='#4C72B0', linewidth=1.5),
                    capprops=dict(color='#4C72B0', linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='#D55E00', 
                                    markersize=4, alpha=0.5))

ax1.set_yticks(range(1, len(topic_labels) + 1))
ax1.set_yticklabels(topic_labels, fontsize=9)
ax1.set_xlabel('Transfer Duration (ms)', fontsize=11)
ax1.set_title(f'Message Transfer: Message Published (rcl_publish) → Message Taken (rmw_take) \n{len(transfer_durations)} Runs', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Add legend
legend_elements = [
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='blue', 
            markersize=6, label='Mean'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D55E00', 
            markersize=6, alpha=0.5, label='Outliers')
]
ax1.legend(handles=legend_elements, loc='lower right')

# Plot 2: Processing Durations Boxplot
# Structure: processing_durations = {timestamp: {publisher_topic: {subscription_topic: duration_ns}}}
# Aggregate durations per topic pair across all test runs
pair_durations_map = {}
for timestamp, publishers_dict in processing_durations.items():
    for publisher_topic, subscriptions_dict in publishers_dict.items():
        for subscription_topic, duration_ns in subscriptions_dict.items():
            pair_key = f"{subscription_topic.split('/')[-1]} → {publisher_topic.split('/')[-1]}"
            if pair_key not in pair_durations_map:
                pair_durations_map[pair_key] = []
            pair_durations_map[pair_key].append(duration_ns / 1e6)  # Convert to ms

# Sort by publisher topic (destination, after the arrow)
sorted_pairs = sorted(pair_durations_map.items(), 
                    key=lambda x: x[0].split(' → ')[1] if ' → ' in x[0] else x[0])

# Limit to top pairs
max_pairs = 15
sorted_pairs = sorted_pairs[:max_pairs]

pair_labels = [pair for pair, _ in sorted_pairs]
boxplot_data2 = [durations for _, durations in sorted_pairs]

# Create boxplot
bp2 = ax2.boxplot(boxplot_data2, vert=False, patch_artist=True,
                    showmeans=True, meanline=False,
                    medianprops=dict(color='red', linewidth=2),
                    meanprops=dict(marker='D', markerfacecolor='blue', markersize=6),
                    boxprops=dict(facecolor='#E74C3C', alpha=0.7),
                    whiskerprops=dict(color='#E74C3C', linewidth=1.5),
                    capprops=dict(color='#E74C3C', linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='#D55E00', 
                                markersize=4, alpha=0.5))

ax2.set_yticks(range(1, len(pair_labels) + 1))
ax2.set_yticklabels(pair_labels, fontsize=9)
ax2.set_xlabel('Processing Duration (ms)', fontsize=11)
ax2.set_title(f'Node Processing: Message Taken (rmw_take) → Message Published (rcl_publish) \n{len(processing_durations)} Runs', 
                fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Add legend
legend_elements = [
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='blue', 
            markersize=6, label='Mean'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D55E00', 
            markersize=6, alpha=0.5, label='Outliers')
]
ax2.legend(handles=legend_elements, loc='lower right')

# Plot 3: Processing Duration from Latest Input to Output
# For each timestamp, find the latest received input and calculate duration to output
latest_input_durations_map = {}
for timestamp, publishers_dict in processing_durations.items():
    for publisher_topic, subscriptions_dict in publishers_dict.items():
        if subscriptions_dict:
            # Find the minimum duration (corresponds to latest received input)
            min_duration_ns = min(subscriptions_dict.values())
            latest_input_topic = [sub for sub, dur in subscriptions_dict.items() if dur == min_duration_ns][0]
            
            label_key = f"{publisher_topic.split('/')[-1]}"
            
            if label_key not in latest_input_durations_map:
                latest_input_durations_map[label_key] = []
            latest_input_durations_map[label_key].append(min_duration_ns / 1e6)  # Convert to ms

# Sort by publisher topic (alphabetically)
sorted_latest = sorted(latest_input_durations_map.items(), key=lambda x: x[0])

# Limit to top pairs
max_latest = 15
sorted_latest = sorted_latest[:max_latest]

latest_labels = [label for label, _ in sorted_latest]
boxplot_data3 = [durations for _, durations in sorted_latest]

# Create boxplot
bp3 = ax3.boxplot(boxplot_data3, vert=False, patch_artist=True,
                    showmeans=True, meanline=False,
                    medianprops=dict(color='red', linewidth=2),
                    meanprops=dict(marker='D', markerfacecolor='blue', markersize=6),
                    boxprops=dict(facecolor='#27AE60', alpha=0.7),
                    whiskerprops=dict(color='#27AE60', linewidth=1.5),
                    capprops=dict(color='#27AE60', linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor='#D55E00', 
                                markersize=4, alpha=0.5))

ax3.set_yticks(range(1, len(latest_labels) + 1))
ax3.set_yticklabels(latest_labels, fontsize=9)
ax3.set_xlabel('Processing Duration (ms)', fontsize=11)
ax3.set_title(f'Node Processing (shortest): Latest Input → Output \n(Minimum processing time after all inputs available) \n{len(processing_durations)} Runs', 
                fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# Add legend
legend_elements = [
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='blue', 
            markersize=6, label='Mean'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D55E00', 
            markersize=6, alpha=0.5, label='Outliers')
]
ax3.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*100)
print("TRANSFER DURATIONS SUMMARY")
print("="*100)
if transfer_durations and topic_durations_map:
    print(f"{'Topic':<60} {'Runs':<8} {'Min (ms)':<12} {'Median (ms)':<12} {'Max (ms)':<12} {'Mean (ms)':<12}")
    print("-"*100)
    for topic, durations in sorted(topic_durations_map.items(), 
                                   key=lambda x: np.median(x[1]), reverse=True):
        print(f"{topic:<60} {len(durations):<8} {np.min(durations):>10.3f}  "
              f"{np.median(durations):>10.3f}  {np.max(durations):>10.3f}  "
              f"{np.mean(durations):>10.3f}")
    print(f"\nTotal topics: {len(topic_durations_map)}, Test runs: {len(transfer_durations)}")
else:
    print("No transfer durations collected yet")

print("\n" + "="*100)
print("PROCESSING DURATIONS SUMMARY")
print("="*100)
if processing_durations and pair_durations_map:
    print(f"{'Topic Pair':<70} {'Runs':<8} {'Min (ms)':<12} {'Median (ms)':<12} {'Max (ms)':<12}")
    print("-"*100)
    for pair, durations in sorted(pair_durations_map.items(), 
                                 key=lambda x: np.median(x[1]), reverse=True):
        print(f"{pair:<70} {len(durations):<8} {np.min(durations):>10.3f}  "
              f"{np.median(durations):>10.3f}  {np.max(durations):>10.3f}")
    print(f"\nTotal topic pairs: {len(pair_durations_map)}, Test runs: {len(processing_durations)}")
else:
    print("No processing durations collected yet")

print("\n" + "="*100)
print("PROCESSING DURATIONS FROM LATEST INPUT TO OUTPUT")
print("="*100)
if processing_durations and latest_input_durations_map:
    print(f"{'Publisher Topic':<70} {'Runs':<8} {'Min (ms)':<12} {'Median (ms)':<12} {'Max (ms)':<12}")
    print("-"*100)
    for label, durations in sorted(latest_input_durations_map.items(), 
                                   key=lambda x: np.median(x[1]), reverse=True):
        print(f"{label:<70} {len(durations):<8} {np.min(durations):>10.3f}  "
              f"{np.median(durations):>10.3f}  {np.max(durations):>10.3f}")
    print(f"\nTotal publishers: {len(latest_input_durations_map)}, Test runs: {len(processing_durations)}")
else:
    print("No processing durations collected yet")

## Analyze Payload Sizes per Topic

Sizes of the ROS messages computed on middleware layer (`rmw_take`). Requires changes in `rmw_zenoh_cpp` from [this branch](https://github.com/RaphvK/rmw_zenoh/tree/jazzy-ika).

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(14, 8))

# Plot 1: Payload Size Boxplot
# Structure: payload_sizes = {timestamp: {topic: size_bytes}}
# Aggregate sizes per topic across all test runs
topic_sizes_map = {}
for timestamp, topics_dict in payload_sizes.items():
    for topic, size_bytes in topics_dict.items():
        if topic not in topic_sizes_map:
            topic_sizes_map[topic] = []
        if size_bytes is not None:
            topic_sizes_map[topic].append(size_bytes)

# Determine appropriate scale based on maximum payload size
max_size = max(max(sizes, default=0) for sizes in topic_sizes_map.values()) if topic_sizes_map else 0
if max_size == 0:
    print("No payload sizes collected yet")
else:
    if max_size >= 1e6:
        scale_factor = 1e6
        size_unit = 'MB'
    elif max_size >= 1e3:
        scale_factor = 1e3
        size_unit = 'KB'
    else:
        scale_factor = 1
        size_unit = 'bytes'

    # Sort topics by median size
    sorted_topics = sorted(topic_sizes_map.items(), 
                        key=lambda x: np.median(x[1]), reverse=True)

    # Limit to top topics to keep visualization readable
    max_topics = 15
    sorted_topics = sorted_topics[:max_topics]

    topic_labels = [topic.split('/')[-1] if len(topic) > 40 else topic 
                    for topic, _ in sorted_topics]
    # Scale the data to the appropriate unit
    boxplot_data = [[size / scale_factor for size in sizes] for _, sizes in sorted_topics]

    # Create boxplot
    bp1 = ax1.boxplot(boxplot_data, vert=False, patch_artist=True,
                        showmeans=True, meanline=False,
                        medianprops=dict(color='red', linewidth=2),
                        meanprops=dict(marker='D', markerfacecolor='blue', markersize=6),
                        boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                        whiskerprops=dict(color='#4C72B0', linewidth=1.5),
                        capprops=dict(color='#4C72B0', linewidth=1.5),
                        flierprops=dict(marker='o', markerfacecolor='#D55E00', 
                                        markersize=4, alpha=0.5))

    ax1.set_yticks(range(1, len(topic_labels) + 1))
    ax1.set_yticklabels(topic_labels, fontsize=9)
    ax1.set_xlabel(f'Payload Size ({size_unit})', fontsize=11)
    ax1.set_title(f'Message Payload Sizes (rmw_take) \n{len(payload_sizes)} Runs', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')

    # Add legend
    legend_elements = [
        Line2D([0], [0], color='red', linewidth=2, label='Median'),
        Line2D([0], [0], marker='D', color='w', markerfacecolor='blue', 
                markersize=6, label='Mean'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#D55E00', 
                markersize=6, alpha=0.5, label='Outliers')
    ]
    ax1.legend(handles=legend_elements, loc='lower right')

    # Print summary statistics
    print("\n" + "="*100)
    print("PAYLOAD SIZES SUMMARY")
    print("="*100)
    if payload_sizes and topic_sizes_map:
        print(f"{'Topic':<60} {'Runs':<8} {'Min (bytes)':<12} {'Median (bytes)':<12} {'Max (bytes)':<12} {'Mean (bytes)':<12}")
        print("-"*100)
        for topic, sizes in sorted(topic_sizes_map.items(), 
                                    key=lambda x: np.median(x[1]), reverse=True):
            if len(sizes) == 0:
                continue
            print(f"{topic:<60} {len(sizes):<8} {np.min(sizes):>10.3f}  "
                f"{np.median(sizes):>10.3f}  {np.max(sizes):>10.3f}  "
                f"{np.mean(sizes):>10.3f}")
        print(f"\nTotal topics: {len(topic_sizes_map)}, Test runs: {len(payload_sizes)}")
    else:
        print("No payload sizes collected yet")